<a href="https://colab.research.google.com/github/sriharikrishna/JDEV-Tutorial/blob/python/02MultipleOutputs/02MultipleOutputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## JAX Translation and Differentiation

First, we'll import `jax` and `jax.numpy` and define the equivalent function in JAX. Unlike C's pass-by-reference for `outcos` and `outsin`, JAX functions can directly return multiple values.

In [ ]:
import jax
import jax.numpy as jnp

def get_radius_jax(x):
    tmp = x**2 / 2.0 + x + 1.0
    outcos = jnp.cos(tmp)
    outsin = jnp.sin(tmp)
    radius_squared = outcos**2 + outsin**2
    return radius_squared, outcos, outsin

Now, let's test the function with an input value, similar to the `main` function in the C code.

In [ ]:
innum = 2.0 # Example input, similar to atof(argv[1])

rad_sq, outcos_val, outsin_val = get_radius_jax(innum)

print(f"Radius Squared = {rad_sq:.6f}\t -- \t cos = {outcos_val:.6f} - sin = {outsin_val:.6f}")

Radius Squared = 1.000000	 -- 	 cos = 0.283662 - sin = -0.958924


The original problem statement mentions differentiating many dependent variables with respect to a single independent variable. We can use `jax.grad` to compute the gradients of `radius_squared`, `outcos`, and `outsin` with respect to the input `x`.

In [ ]:
# Define helper functions to extract the specific output for differentiation
def get_radius_squared(x):
    return get_radius_jax(x)[0]

def get_outcos(x):
    return get_radius_jax(x)[1]

def get_outsin(x):
    return get_radius_jax(x)[2]

# Compute gradients
grad_rad_sq = jax.grad(get_radius_squared)
grad_outcos = jax.grad(get_outcos)
grad_outsin = jax.grad(get_outsin)

# Evaluate gradients at the input number
print(f"Gradient of Radius Squared w.r.t. input: {grad_rad_sq(innum):.6f}")
print(f"Gradient of outcos w.r.t. input: {grad_outcos(innum):.6f}")
print(f"Gradient of outsin w.r.t. input: {grad_outsin(innum):.6f}")

Gradient of Radius Squared w.r.t. input: 0.000000
Gradient of outcos w.r.t. input: 2.876773
Gradient of outsin w.r.t. input: 0.850987


### Using `jax.vjp` for Gradients

`jax.vjp` returns a pair: the function's output and a 'vector-Jacobian product' function (`vjp_fun`). We can then call `vjp_fun` with a 'cotangent' vector (often called `g` or `grad_output`) to get the gradients of the original inputs with respect to `jnp.dot(output, cotangent_vector)`.

In [ ]:
# Apply jax.vjp to the original function
y, vjp_fun = jax.vjp(get_radius_jax, innum)

# The 'y' here will be a tuple: (radius_squared, outcos, outsin)
# The cotangent vector 'g' must have the same shape as 'y'.
# To get the gradient of each component, we'll pass a 'one-hot' vector.

# Gradient of radius_squared (the first component, index 0)
# We pass a cotangent vector (1.0, 0.0, 0.0) indicating we want the gradient
# with respect to the first output.
grad_rad_sq_vjp = vjp_fun((jnp.array(1.0), jnp.array(0.0), jnp.array(0.0)))[0]

# Gradient of outcos (the second component, index 1)
# We pass a cotangent vector (0.0, 1.0, 0.0)
grad_outcos_vjp = vjp_fun((jnp.array(0.0), jnp.array(1.0), jnp.array(0.0)))[0]

# Gradient of outsin (the third component, index 2)
# We pass a cotangent vector (0.0, 0.0, 1.0)
grad_outsin_vjp = vjp_fun((jnp.array(0.0), jnp.array(0.0), jnp.array(1.0)))[0]

print("--- Gradients using jax.vjp ---")
print(f"Gradient of Radius Squared w.r.t. input: {grad_rad_sq_vjp:.6f}")
print(f"Gradient of outcos w.r.t. input: {grad_outcos_vjp:.6f}")
print(f"Gradient of outsin w.r.t. input: {grad_outsin_vjp:.6f}")

print("\n--- Comparing with jax.grad results (should be identical) ---")
print(f"jax.grad for Radius Squared: {grad_rad_sq(innum):.6f}")
print(f"jax.grad for outcos: {grad_outcos(innum):.6f}")
print(f"jax.grad for outsin: {grad_outsin(innum):.6f}")

--- Gradients using jax.vjp ---
Gradient of Radius Squared w.r.t. input: 0.000000
Gradient of outcos w.r.t. input: 2.876773
Gradient of outsin w.r.t. input: 0.850987

--- Comparing with jax.grad results (should be identical) ---
jax.grad for Radius Squared: 0.000000
jax.grad for outcos: 2.876773
jax.grad for outsin: 0.850987


### Using `jax.jvp` for Forward-Mode Differentiation

`jax.jvp` takes a function, its primal arguments (the input values), and tangent arguments (the direction along which to differentiate). For a scalar input like `innum`, the tangent argument is typically `(1.0,)` to compute the standard first derivative.

The output of `jax.jvp` is `(primals_out, tangents_out)`, where `primals_out` are the function's normal outputs and `tangents_out` are the derivatives of each output with respect to the input `x`.

In [ ]:
# Define a wrapper for get_radius_jax that returns a tuple of its outputs
def get_radius_jax_multi_output(x):
    rad_sq, outcos, outsin = get_radius_jax(x)
    return (rad_sq, outcos, outsin)

# Compute JVP. The tangent `(1.0,)` indicates we want the derivative with respect to `innum`.
primals_out, tangents_out = jax.jvp(get_radius_jax_multi_output, (innum,), (jnp.array(1.0),))

# tangents_out will be a tuple containing the derivatives of each original output
jvp_grad_rad_sq, jvp_grad_outcos, jvp_grad_outsin = tangents_out

print("--- Gradients using jax.jvp (Forward-Mode) ---")
print(f"Primal outputs: Radius Squared = {primals_out[0]:.6f}, cos = {primals_out[1]:.6f}, sin = {primals_out[2]:.6f}")
print(f"JVP Gradient of Radius Squared w.r.t. input: {jvp_grad_rad_sq:.6f}")
print(f"JVP Gradient of outcos w.r.t. input: {jvp_grad_outcos:.6f}")
print(f"JVP Gradient of outsin w.r.t. input: {jvp_grad_outsin:.6f}")

print("\n--- Comparing with jax.grad results (should be identical) ---")
print(f"jax.grad for Radius Squared: {grad_rad_sq(innum):.6f}")
print(f"jax.grad for outcos: {grad_outcos(innum):.6f}")
print(f"jax.grad for outsin: {grad_outsin(innum):.6f}")

--- Gradients using jax.jvp (Forward-Mode) ---
Primal outputs: Radius Squared = 1.000000, cos = 0.283662, sin = -0.958924
JVP Gradient of Radius Squared w.r.t. input: 0.000000
JVP Gradient of outcos w.r.t. input: 2.876773
JVP Gradient of outsin w.r.t. input: 0.850987

--- Comparing with jax.grad results (should be identical) ---
jax.grad for Radius Squared: 0.000000
jax.grad for outcos: 2.876773
jax.grad for outsin: 0.850987
